# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "DEBUG"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda


In [4]:
from mllm_shap.connectors import ModelConfig, TransformersCausalText
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import PowerShiftNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define LiquidAudio model (this call loads it up to the memory!).

Create compact explainer that will make initial call and then explain it using Shapley Values approximated using Monte Carlo. Set minimal number of samples for demo purpose, that is only first-order omission ones (where only one token at the time is hidden) and empty sample (if it is not "globally" empty, that is when expandability is not performed on all tokens).

PowerNormalizer first shifts all shapley values by subtracting their minimum (so new minimal value will be 0.0), then raises them to power of 2.0 and normalizes so they sum to 1.0.

In [5]:
model = TransformersCausalText(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = McShapExplainer(
    fraction=0.7,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=PowerShiftNormalizer(power=2.0),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Create new chat that treats Assistant messages as system, that is will ignore them for shapley values calculation - they will be feed to each prompt as system messages. This significantly reduces number of requests needed for multi turn expandability, yet might not be possible due to business requirements. 

To further reduce number of calls we exclude punctuation tokens.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,
    token_filter=ExcludePunctuationTokensFilter(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.SYSTEM)
chat.add_text("You are a helpful assistant that answers questions briefly.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Explain the process of photosynthesis.")
chat.end_turn()

2025-12-01 22:57:27,682 - mllm_shap.connectors.base.chat - DEBUG - New turn started with speaker: SYSTEM
2025-12-01 22:57:27,685 - mllm_shap.connectors.base.chat - DEBUG - Extending token turns: 10
2025-12-01 22:57:27,686 - mllm_shap.connectors.base.chat - DEBUG - Extending token roles: 10
2025-12-01 22:57:27,687 - mllm_shap.connectors.base.chat - DEBUG - Extending text tokens (no system mask): 10
2025-12-01 22:57:27,687 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 22:57:27,687 - mllm_shap.connectors.base.chat - DEBUG - Added text: 'You are a helpful assistant that answers questions briefly.' (is_system=True, speaker=SYSTEM)
2025-12-01 22:57:27,688 - mllm_shap.connectors.base.chat - DEBUG - Turn ended for speaker: SYSTEM
2025-12-01 22:57:27,688 - mllm_shap.connectors.base.chat - DEBUG - New turn started with speaker: USER
2025-12-01 22:57:27,688 - mllm_shap.connectors.base.chat - DEBUG - Extending token turns: 8
2025-12-01 

Let's have a look at chat representation:

In [7]:
chat.get_conversation()

2025-12-01 22:57:27,763 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([1639], device='cuda:0')
2025-12-01 22:57:27,765 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([389], device='cuda:0')
2025-12-01 22:57:27,765 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([257], device='cuda:0')
2025-12-01 22:57:27,766 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7613], device='cuda:0')
2025-12-01 22:57:27,766 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([8796], device='cuda:0')
2025-12-01 22:57:27,767 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([326], device='cuda:0')
2025-12-01 22:57:27,767 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7429], device='cuda:0')
2025-12-01 22:57:27,767 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([2683], device='cuda:0')
202

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=None)],
 [ChatEntry(content_type=0, roles=[USER, USER, ..., USER, USER], content='Expl, ain,  the,  process,  of,  photos, ynthesis, ....', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 64 tokens and change text_temperature from default 0.0 to 0.2. 

In [ ]:
print(f"Current process ID: {os.getpid()}")

Current process ID: 68436


In [18]:
generation_kwargs = {"max_new_tokens": 50, "model_config": ModelConfig(text_temperature=0.0, text_top_k=1)}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-12-01 23:04:38,430 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-12-01 23:04:38,431 - mllm_shap.connectors.base.model - DEBUG - Generating audio with max_new_tokens=50, keep_history=True
/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/compact.py:57: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2025-12-01 23:04:38,442 - mllm_shap.connectors.base.chat - DEBUG - New turn started with speaker: ASSISTANT
2025-12-01 23:04:39,556 - mllm_shap.connectors.base.model - DEBUG - Setting chat history with text tokens (1), audio tokens (0), modality flags (50).
2025-12-01 23:04:39,557 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 23:04:39,557 - mllm_shap.connectors.base.chat - DEBUG - Extending token turns: 50
2025-12-01 23:04:39,558 - mllm_shap.con

Calculating SHAP values:   0%|          | 0/88 [00:00<?, ?it/s]

2025-12-01 23:04:39,580 - mllm_shap.shap.base.shap_explainer - DEBUG - Generated zero or all-ones mask, skipping.
2025-12-01 23:04:39,580 - mllm_shap.shap.base._generate_responses - DEBUG - Processing mask tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
        False,  True,  True,  True,  True,  True,  True,  True],
       device='cuda:0')
2025-12-01 23:04:39,581 - mllm_shap.connectors.base.chat - DEBUG - Creating new chat instance from existing chat with masks.
2025-12-01 23:04:39,583 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 23:04:39,583 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=False, shap=True).
2025-12-01 23:04:39,584 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=False, shap=True).
2025-12-01 23:04:39,584 - mllm_shap.connectors.base.model - DEBUG - Generating audio with max_new_tokens=50, keep_history=True
/home/mvishi

Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

Cache object stores actual shapley values as well as calculated embeddings and masks. They will be reused in next call regardless to the method, so for monte-carlo it is just larger sample, for precise it means some results might get excluded.

Let's first analyze history - it is a list of size equivalent to number of calls made for calculations + 1 (first entry, for base calculations, always None) - in this case, 4 (3 for base one-versus-all and one for empty call). Each entry is a tuple of following values:

- mask for that entry
- mash hash
- source chat with masked entry or None if corresponding mask was available in cache
- model response object

or None - when either corresponding mask was extracted from cache or it has risen an AllTextTokensFilteredOutError error.

Let's see all chats that were taken into account:

In [ ]:
print([c[2].decode_text() if c is not None else None for c in result.history])

2025-12-01 22:58:12,150 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (17): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
          391,   262,  1429,   286,  5205, 44411,    13], device='cuda:0')
2025-12-01 22:58:12,151 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (17): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
        18438,   262,  1429,   286,  5205, 44411,    13], device='cuda:0')
2025-12-01 22:58:12,151 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (17): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
        18438,   391,  1429,   286,  5205, 44411,    13], device='cuda:0')
2025-12-01 22:58:12,152 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (17): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
        18438,   391,   262,   286,  5205, 44411,    13], device='cuda:0')
2025-12-01 2

['You are a helpful assistant that answers questions briefly.ain the process of photosynthesis.',
 'You are a helpful assistant that answers questions briefly.Expl the process of photosynthesis.',
 'You are a helpful assistant that answers questions briefly.Explain process of photosynthesis.',
 'You are a helpful assistant that answers questions briefly.Explain the of photosynthesis.',
 'You are a helpful assistant that answers questions briefly.Explain the process photosynthesis.',
 'You are a helpful assistant that answers questions briefly.Explain the process ofynthesis.',
 'You are a helpful assistant that answers questions briefly.Explain the process of photos.',
 'You are a helpful assistant that answers questions briefly.Explain of photos.',
 'You are a helpful assistant that answers questions briefly.Explain photos.',
 'You are a helpful assistant that answers questions briefly.Explain process.',
 'You are a helpful assistant that answers questions briefly.ain the.',
 'You are 

We can see that "?" was never removed, as it is present even in the empty mask. For rest, we can see that all system tokens are always present and only user tokens get masked. out between each calls.

Let's now analyze calculated shapley values.

In [11]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

2025-12-01 22:58:12,282 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([1639], device='cuda:0')
2025-12-01 22:58:12,283 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([389], device='cuda:0')
2025-12-01 22:58:12,284 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([257], device='cuda:0')
2025-12-01 22:58:12,284 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7613], device='cuda:0')
2025-12-01 22:58:12,285 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([8796], device='cuda:0')
2025-12-01 22:58:12,285 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([326], device='cuda:0')
2025-12-01 22:58:12,285 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7429], device='cuda:0')
2025-12-01 22:58:12,286 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([2683], device='cuda:0')
202

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, ..., USER, USER], content='Expl, ain,  the,  process,  of,  photos, ynthesis, ....', shap_values=[0.17809191346168518, 0.003641324583441019, ..., 0.43905940651893616, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Output, :,  Photos, ynthesis,  is,  the,  process,  by,  which,  plants,  use,  light,  energy, ...', shap_values=[nan, nan, ..., nan, nan])]]


Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [12]:
user_entry = explained_chat_conversation[1][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,Expl,0.178092,0
1,ain,0.003641,0
2,the,0.000000,0
3,process,0.010230,0
4,of,0.008405,0
5,photos,0.360573,0
6,ynthesis,0.439059,0
7,.,nan,0


Let's create another turn to see how input significance will change:

In [13]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

2025-12-01 22:58:12,365 - mllm_shap.connectors.base.chat - DEBUG - New turn started with speaker: USER
2025-12-01 22:58:12,366 - mllm_shap.connectors.base.chat - DEBUG - Extending token turns: 4
2025-12-01 22:58:12,367 - mllm_shap.connectors.base.chat - DEBUG - Extending token roles: 4
2025-12-01 22:58:12,367 - mllm_shap.connectors.base.chat - DEBUG - Extending text tokens (no system mask): 4
2025-12-01 22:58:12,368 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 22:58:12,368 - mllm_shap.connectors.base.chat - DEBUG - Added text: 'Can you repeat?' (is_system=False, speaker=USER)
2025-12-01 22:58:12,369 - mllm_shap.connectors.base.chat - DEBUG - Turn ended for speaker: USER


And again, let's explain it:

In [14]:
result = explainer(chat=explained_chat, verbose=True, generation_kwargs=generation_kwargs)

2025-12-01 22:58:12,409 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-12-01 22:58:12,409 - mllm_shap.connectors.base.model - DEBUG - Generating audio with max_new_tokens=20, keep_history=True
/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/compact.py:57: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2025-12-01 22:58:12,427 - mllm_shap.connectors.base.chat - DEBUG - New turn started with speaker: ASSISTANT
2025-12-01 22:58:12,561 - mllm_shap.connectors.base.model - DEBUG - Setting chat history with text tokens (1), audio tokens (0), modality flags (2).
2025-12-01 22:58:12,561 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 22:58:12,562 - mllm_shap.connectors.base.chat - DEBUG - Extending token turns: 2
2025-12-01 22:58:12,562 - mllm_shap.conne

Calculating SHAP values:   0%|          | 0/716 [00:00<?, ?it/s]

2025-12-01 22:58:12,577 - mllm_shap.shap.base.shap_explainer - DEBUG - Generated zero or all-ones mask, skipping.
2025-12-01 22:58:12,577 - mllm_shap.shap.base._generate_responses - DEBUG - Processing mask tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
        False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True], device='cuda:0')
2025-12-01 22:58:12,578 - mllm_shap.connectors.base.chat - DEBUG - Creating new chat instance from existing chat with masks.
2025-12-01 22:58:12,579 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=True, shap=False).
2025-12-01 22:58:12,579 - mllm_shap.connectors.base.chat - DEBUG - Refreshing cached properties (full=False, shap=True).
2025-12-01 22:58:12,580 - mllm_shap.connectors.base.chat - DEBUG - Refreshi

In [ ]:
print([c[2].decode_text() if c is not None else None for c in result.history])

2025-12-01 23:00:32,075 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (41): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
          391,   262,  1429,   286,  5205, 44411,    13,   198, 26410,    25,
         9434, 44411,   318,   262,  1429,   416,   543,  6134,   779,  1657,
         2568,   284, 10385,  1660,   290,  6588, 17556,  6090,   345,  9585,
           30], device='cuda:0')
2025-12-01 23:00:32,077 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (41): tensor([ 1639,   389,   257,  7613,  8796,   326,  7429,  2683, 11589,    13,
        18438,   262,  1429,   286,  5205, 44411,    13,   198, 26410,    25,
         9434, 44411,   318,   262,  1429,   416,   543,  6134,   779,  1657,
         2568,   284, 10385,  1660,   290,  6588, 17556,  6090,   345,  9585,
           30], device='cuda:0')
2025-12-01 23:00:32,079 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (41): tensor([ 1639,   389,   257,

['You are a helpful assistant that answers questions briefly.ain the process of photosynthesis.\nOutput: Photosynthesis is the process by which plants use light energy to convert water and carbon dioxideCan you repeat?',
 'You are a helpful assistant that answers questions briefly.Expl the process of photosynthesis.\nOutput: Photosynthesis is the process by which plants use light energy to convert water and carbon dioxideCan you repeat?',
 'You are a helpful assistant that answers questions briefly.Explain process of photosynthesis.\nOutput: Photosynthesis is the process by which plants use light energy to convert water and carbon dioxideCan you repeat?',
 'You are a helpful assistant that answers questions briefly.Explain the of photosynthesis.\nOutput: Photosynthesis is the process by which plants use light energy to convert water and carbon dioxideCan you repeat?',
 'You are a helpful assistant that answers questions briefly.Explain the process photosynthesis.\nOutput: Photosynthesi

In [16]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

2025-12-01 23:00:33,230 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([1639], device='cuda:0')
2025-12-01 23:00:33,232 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([389], device='cuda:0')
2025-12-01 23:00:33,234 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([257], device='cuda:0')
2025-12-01 23:00:33,236 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7613], device='cuda:0')
2025-12-01 23:00:33,238 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([8796], device='cuda:0')
2025-12-01 23:00:33,239 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([326], device='cuda:0')
2025-12-01 23:00:33,241 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([7429], device='cuda:0')
2025-12-01 23:00:33,243 - mllm_shap.connectors.base.chat - DEBUG - Decoding text tokens (1): tensor([2683], device='cuda:0')
202

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, ..., USER, USER], content='Expl, ain,  the,  process,  of,  photos, ynthesis, ....', shap_values=[0.04690830036997795, 0.058501821011304855, ..., 0.10175488889217377, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Output, :,  Photos, ynthesis,  is,  the,  process,  by,  which,  plants,  use,  light,  energy, ...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Can,  you,  repeat, ?', shap_values=[0.32077863812446594, 0.0, 0.1437297761440277, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT], content='\n, <|endoftext|>', shap_values=[nan, nan])]]


In [17]:
dt = []
for i in (1, 3):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,Expl,0.046908,0,1
1,ain,0.058502,0,1
2,the,0.095018,0,1
3,process,0.071393,0,1
4,of,0.060739,0,1
5,photos,0.101177,0,1
6,ynthesis,0.101755,0,1
7,.,nan,0,1
8,Can,0.320779,0,3
9,you,0.000000,0,3
